# Coarse articulatory activity from GRU-assisted timing

## Why start here

T12 provides phoneme sequences but no independent phoneme times. Before training another decoder, this notebook asks whether the strongest completed local GRU is already accurate enough after its ordinary CTC phoneme predictions are mapped into broader articulatory groups, then tests whether reference-aligned consonant articulators are linearly accessible from its hidden state. These are decoder-assisted diagnostics, not independent timing evidence.

The primary grouping is **vowel / consonant / silence-or-word-boundary**. A secondary manner grouping separates stops, fricatives, affricates, nasals, liquids, and glides. CTC blank remains a separate alignment state and is never treated as silence.

The primary comparison is native phoneme error rate (PER) versus post-hoc grouped token error rate. References and greedy CTC predictions are decoded at the phoneme level first and only then mapped to groups. Repeated adjacent groups are deliberately preserved; collapsing them after mapping would erase real phoneme tokens and make the grouped score artificially optimistic.

The selected GRU is the completed Brain-to-Text 2024 TX+SBP run whose saved notebook output records the best local GRU validation PER found in the repository: **0.32515 at step 18,300**. The older curated result ledger reports 0.37485. This notebook verifies the checkpoint and its companion summary rather than trusting the historical notebook output alone. The local GRU is an LLM-assisted PyTorch port/adaptation of likely Willett/Stanford TensorFlow source; it is not an official or verified reproduction.

The first linear probe uses the canonical articulatory taxonomy, mean-pools hidden states over reference-constrained CTC spans, fits independent lips/tongue-front/tongue-body logistic probes on the first 20 sessions, and evaluates the final four sessions. The probe head is session-held-out; the frozen GRU is not, because its adapters and checkpoint selection used data from all 24 sessions.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import importlib.util
import os
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path('/content/utah-ssl')
REPO_URL = 'https://github.com/ethan-read/utah-ssl.git'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', 'main'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=REPO_DIR, check=True)

required = {'numpy': 'numpy', 'pandas': 'pandas', 'torch': 'torch', 'matplotlib': 'matplotlib', 'sklearn': 'scikit-learn'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)

os.environ['PYTHONPATH'] = str(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## Checkpoint and evaluation contract

The GRU is run on all 24 Brain-to-Text 2024 `competition_test` sessions. The grouped-sequence diagnostic remains focused on the final four sessions; the linear probe trains on events from the first 20 sessions and tests on the final four. Those labels participated in GRU checkpoint selection, and the GRU has learned adapters for every session, so only the newly fitted probe head is session-held-out. The checkpoint consumes the exact legacy raw TX+SBP cache and training-only global normalization contract with which it was trained; silently substituting the newer SBP-only cache would be incompatible.


In [ ]:
import json
import torch

from experiments.supervised_baselines.checkpointing import config_from_checkpoint
from experiments.supervised_baselines.data import build_willett_problem

DRIVE_ROOT = Path('/content/drive/MyDrive/utah_ssl')
RUN_DIR = DRIVE_ROOT / 'outputs/willett_reconstruction/willett_tx_only_area6v_colab'
CHECKPOINT_PATH = RUN_DIR / 'checkpoint_best.pt'
SUMMARY_PATH = RUN_DIR / 'summary.json'
CACHE_ROOT = DRIVE_ROOT / 'data/cache_v1'
STATS_PATH = DRIVE_ROOT / 'data/stats/split_feature_stats/raw/brain2text24/competition_train/tx_sbp/global_v1.pt'
OUTPUT_DIR = DRIVE_ROOT / 'outputs/neural_trajectories/gru_articulator_probe_b2t24_chronological_v1'
EXPORT_ROOT = OUTPUT_DIR / 'prediction_export'
MODEL_KEY = 'gru_best_step18300_all_val_sessions'
PROBE_TEST_SESSION_IDS = (
    't12.2022.08.13',
    't12.2022.08.18',
    't12.2022.08.23',
    't12.2022.08.25',
)
EXPECTED_BEST_STEP = 18_300
EXPECTED_RECORDED_PER = 0.3251455228501837
OVERWRITE_EXPORT = False
OVERWRITE_RESULTS = False
RESULT_COMPLETION_MARKER = OUTPUT_DIR / '_SUCCESS'
if RESULT_COMPLETION_MARKER.exists():
    if not OVERWRITE_RESULTS:
        raise FileExistsError(f'Completed result already exists: {OUTPUT_DIR}. Set OVERWRITE_RESULTS=True to replace it.')
    RESULT_COMPLETION_MARKER.unlink()

for required_path in (CHECKPOINT_PATH, SUMMARY_PATH, STATS_PATH, STATS_PATH.with_suffix('.json'), CACHE_ROOT / 'brain2text24/metadata.json', CACHE_ROOT / 'brain2text24/manifest.jsonl'):
    assert required_path.exists(), f'Missing required artifact: {required_path}'

summary = json.loads(SUMMARY_PATH.read_text())
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
checkpoint_config = config_from_checkpoint(checkpoint)
recorded_per = float(summary['best_metrics']['val_phoneme_error_rate'])
assert int(summary['best_step']) == EXPECTED_BEST_STEP, summary['best_step']
assert int(checkpoint.get('step', -1)) == EXPECTED_BEST_STEP, checkpoint.get('step')
assert abs(recorded_per - EXPECTED_RECORDED_PER) < 1e-12, recorded_per
assert checkpoint_config.dataset == 'brain2text24'
assert checkpoint_config.decoder_backbone_type == 'gru'
assert checkpoint_config.feature_mode == 'tx_sbp'
assert checkpoint_config.split_policy == 'competition_train_test'
assert checkpoint_config.normalization_mode == 'global'
assert checkpoint_config.patch_size == 14 and checkpoint_config.patch_stride == 4
assert Path(checkpoint_config.cache_root) == CACHE_ROOT

problem = build_willett_problem(
    cache_root=CACHE_ROOT,
    dataset=checkpoint_config.dataset,
    feature_mode=checkpoint_config.feature_mode,
    boundary_key_mode=checkpoint_config.boundary_key_mode,
    split_policy=checkpoint_config.split_policy,
    cv_num_folds=checkpoint_config.cv_num_folds,
    cv_fold_index=checkpoint_config.cv_fold_index,
)
signal = problem['signal_spec']
metadata = problem['metadata']
assert signal.mode == 'tx_sbp' and signal.tx_dim == 128 and signal.sbp_dim == 128
assert signal.column_start == 0 and signal.missing_channel_policy == 'error'
assert int(metadata.get('bin_size_ms', 20)) == 20
assert problem['train_split_name'] == 'competition_train'
assert problem['val_split_name'] == 'competition_test'
ALL_VAL_SESSION_IDS = tuple(problem['val_session_ids'])
PROBE_TRAIN_SESSION_IDS = tuple(session_id for session_id in ALL_VAL_SESSION_IDS if session_id not in PROBE_TEST_SESSION_IDS)
assert len(ALL_VAL_SESSION_IDS) == 24
assert len(PROBE_TRAIN_SESSION_IDS) == 20 and PROBE_TRAIN_SESSION_IDS[-1] == 't12.2022.08.11'
assert ALL_VAL_SESSION_IDS[-4:] == PROBE_TEST_SESSION_IDS
assert set(PROBE_TEST_SESSION_IDS).issubset(set(problem['train_session_ids']))

checkpoint_contract = {
    'checkpoint': str(CHECKPOINT_PATH),
    'recorded_best_step': int(summary['best_step']),
    'recorded_best_validation_per': recorded_per,
    'dataset': checkpoint_config.dataset,
    'signal_spec': signal.to_dict(),
    'cache_root': str(CACHE_ROOT),
    'training_only_global_stats': str(STATS_PATH),
    'normalization': checkpoint_config.normalization_mode,
    'selection_split': problem['val_split_name'],
    'export_sessions': list(ALL_VAL_SESSION_IDS),
    'probe_train_sessions': list(PROBE_TRAIN_SESSION_IDS),
    'probe_test_sessions': list(PROBE_TEST_SESSION_IDS),
    'patch_ms': checkpoint_config.patch_size * 20,
    'stride_ms': checkpoint_config.patch_stride * 20,
}
print(json.dumps(checkpoint_contract, indent=2))
del checkpoint


## Phoneme groups

`SIL` in this vocabulary is a silence/word-boundary target. It is not the CTC blank. The broad grouping is the first decision point; manner groups are a harder secondary check.


In [ ]:
import pandas as pd

from experiments.manifolds.representation_export import (
    PHONEME_CATEGORY_BY_SYMBOL,
    id_to_symbol_from_vocab,
)

vocab = dict(problem['vocab'])
id_to_symbol = id_to_symbol_from_vocab(vocab)
blank_index = int(vocab['blank_index'])
sil_index = int(vocab['sil_index'])
assert id_to_symbol[blank_index] == 'BLANK'
assert id_to_symbol[sil_index] == 'SIL'

MANNER_BY_SYMBOL = dict(PHONEME_CATEGORY_BY_SYMBOL)
BROAD_BY_SYMBOL = {
    symbol: (
        'blank' if manner == 'blank' else
        'silence_boundary' if manner == 'silence' else
        'vowel' if manner == 'vowel' else
        'consonant'
    )
    for symbol, manner in MANNER_BY_SYMBOL.items()
}
MANNER_BY_SYMBOL['SIL'] = 'silence_boundary'

expected_symbols = set(id_to_symbol.values())
assert expected_symbols == set(MANNER_BY_SYMBOL), (expected_symbols - set(MANNER_BY_SYMBOL), set(MANNER_BY_SYMBOL) - expected_symbols)
assert expected_symbols == set(BROAD_BY_SYMBOL)
assert BROAD_BY_SYMBOL['BLANK'] == 'blank' and BROAD_BY_SYMBOL['SIL'] == 'silence_boundary'

group_definition = pd.DataFrame([
    {
        'phoneme_id': token_id,
        'symbol': symbol,
        'broad_group': BROAD_BY_SYMBOL[symbol],
        'manner_group': MANNER_BY_SYMBOL[symbol],
    }
    for token_id, symbol in sorted(id_to_symbol.items())
])
display(group_definition)


## Run the frozen GRU

The repository exporter reconstructs the model from the checkpoint, applies its training-compatible global normalization and inference-time smoothing, runs greedy CTC decoding, and saves both per-frame logits and per-example native phoneme sequences. No labels are supplied to the decoder.


In [ ]:
from experiments.manifolds.representation_export import (
    RepresentationExportConfig,
    export_willett_representations,
)

export_metadata = export_willett_representations(
    RepresentationExportConfig(
        checkpoint_path=CHECKPOINT_PATH,
        export_root=EXPORT_ROOT,
        model_key=MODEL_KEY,
        split='val',
        allowed_session_ids=ALL_VAL_SESSION_IDS,
        max_examples=None,
        batch_size=32,
        shard_size_tokens=50_000,
        bin_size_ms=20,
        overwrite=OVERWRITE_EXPORT,
        device=None,
        repo_dir=REPO_DIR,
        cache_root_override=CACHE_ROOT,
        precomputed_split_stats_path_override=STATS_PATH,
        save_input_windows=False,
    )
)
export_dir = EXPORT_ROOT / MODEL_KEY
stored_export_config = export_metadata['representation_export_config']
assert Path(export_metadata['checkpoint_path']) == CHECKPOINT_PATH
assert int(export_metadata['checkpoint_step']) == EXPECTED_BEST_STEP
assert export_metadata['dataset'] == 'brain2text24'
assert export_metadata['feature_mode'] == 'tx_sbp'
assert export_metadata['export_split'] == 'val'
assert Path(export_metadata['cache_root']) == CACHE_ROOT
assert Path(export_metadata['precomputed_split_stats_path']) == STATS_PATH
assert int(export_metadata['patch_size_bins']) == checkpoint_config.patch_size
assert int(export_metadata['patch_stride_bins']) == checkpoint_config.patch_stride
assert int(export_metadata['bin_size_ms']) == 20
assert tuple(stored_export_config['allowed_session_ids']) == ALL_VAL_SESSION_IDS
assert stored_export_config['max_examples'] is None
assert int(export_metadata['example_count']) == len(problem['val_rows'])
assert int(export_metadata['token_count']) == sum(int(shard['token_count']) for shard in export_metadata['shards'])
assert all(int(shard['hidden_dim']) == int(export_metadata['hidden_dim']) for shard in export_metadata['shards'])
assert all(int(shard['vocab_size']) == int(export_metadata['vocab']['num_classes']) for shard in export_metadata['shards'])
print(json.dumps({key: export_metadata[key] for key in ('checkpoint_step', 'example_count', 'token_count', 'patch_size_ms', 'patch_stride_ms')}, indent=2))


## Native predictions, then grouped predictions

The denominator stays equal to the number of reference phoneme tokens for every scheme. Mapping changes which substitutions count as errors but does not delete repeated tokens. Insertions and deletions therefore remain visible.


In [ ]:
from utah_ssl.ctc import edit_counts

examples = pd.read_csv(export_dir / 'examples.csv', keep_default_na=False)
assert set(examples['session_id']) == set(ALL_VAL_SESSION_IDS)
assert set(examples['source_split']) == {'competition_test'}
assert examples['example_id'].is_unique
group_examples = examples[examples['session_id'].isin(PROBE_TEST_SESSION_IDS)].copy()

def parse_id_sequence(value):
    text = str(value).strip()
    return [] if not text else [int(token) for token in text.split()]

def map_sequence(token_ids, symbol_to_group):
    symbols = [id_to_symbol[int(token_id)] for token_id in token_ids]
    assert 'BLANK' not in symbols, 'CTC-decoded/reference sequences must not contain blank.'
    return [symbol_to_group[symbol] for symbol in symbols]

def score_one(reference, prediction):
    insertions, deletions, substitutions = edit_counts(reference, prediction)
    return {
        'insertions': int(insertions),
        'deletions': int(deletions),
        'substitutions': int(substitutions),
        'errors': int(insertions + deletions + substitutions),
        'reference_tokens': int(len(reference)),
        'predicted_tokens': int(len(prediction)),
    }

# Mapping P B -> consonant consonant must preserve two tokens.
assert map_sequence([27, 7], BROAD_BY_SYMBOL) == ['consonant', 'consonant']
assert score_one(['consonant', 'consonant'], ['consonant'])['deletions'] == 1

schemes = {
    'native_phoneme': None,
    'broad_3way': BROAD_BY_SYMBOL,
    'manner_8way': MANNER_BY_SYMBOL,
}
trial_rows = []
for row in group_examples.itertuples(index=False):
    native_reference = parse_id_sequence(row.reference_ids)
    native_prediction = parse_id_sequence(row.prediction_ids)
    for scheme_name, mapping in schemes.items():
        if mapping is None:
            reference, prediction = native_reference, native_prediction
        else:
            reference = map_sequence(native_reference, mapping)
            prediction = map_sequence(native_prediction, mapping)
        # Do not collapse adjacent equal groups here. Each element still
        # represents one native phoneme token.
        trial_rows.append({
            'example_id': row.example_id,
            'session_id': row.session_id,
            'scheme': scheme_name,
            **score_one(reference, prediction),
        })
trial_metrics = pd.DataFrame(trial_rows)

def aggregate_scores(frame, group_columns):
    sums = frame.groupby(group_columns, as_index=False)[
        ['insertions', 'deletions', 'substitutions', 'errors', 'reference_tokens', 'predicted_tokens']
    ].sum()
    sums['error_rate'] = sums['errors'] / sums['reference_tokens']
    sums['predicted_to_reference_ratio'] = sums['predicted_tokens'] / sums['reference_tokens']
    return sums

per_session_metrics = aggregate_scores(trial_metrics, ['session_id', 'scheme'])
pooled_metrics = aggregate_scores(trial_metrics, ['scheme'])
display(pooled_metrics)
display(per_session_metrics.pivot(index='session_id', columns='scheme', values='error_rate').reset_index())


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

tokens = pd.read_csv(export_dir / 'tokens.csv')
assert set(tokens['session_id']) == set(ALL_VAL_SESSION_IDS)
group_tokens = tokens[tokens['session_id'].isin(PROBE_TEST_SESSION_IDS)].copy()
nonblank_mass = 1.0 - group_tokens['blank_prob'].to_numpy(dtype=float)
broad_probs = np.column_stack([
    group_tokens['vowel_prob'].to_numpy(dtype=float),
    group_tokens['consonant_prob'].to_numpy(dtype=float),
    group_tokens['silence_prob'].to_numpy(dtype=float),
])
assert np.allclose(broad_probs.sum(axis=1), nonblank_mass, atol=2e-5)
conditional_broad = broad_probs / np.maximum(nonblank_mass[:, None], 1e-12)
high_nonblank = nonblank_mass >= 0.5
posterior_summary = pd.DataFrame([{
    'frames': int(len(group_tokens)),
    'high_nonblank_frames': int(high_nonblank.sum()),
    'high_nonblank_fraction': float(high_nonblank.mean()),
    'median_nonblank_mass': float(np.median(nonblank_mass)),
    'median_top_broad_probability_given_nonblank_high_nonblank_frames': (
        float(np.median(conditional_broad[high_nonblank].max(axis=1))) if high_nonblank.any() else float('nan')
    ),
}])
display(posterior_summary)

plot_frame = per_session_metrics.copy()
fig, ax = plt.subplots(figsize=(10, 4.5))
scheme_order = ['native_phoneme', 'manner_8way', 'broad_3way']
x = np.arange(len(PROBE_TEST_SESSION_IDS))
width = 0.24
for offset, scheme_name in enumerate(scheme_order):
    values = plot_frame[plot_frame.scheme == scheme_name].set_index('session_id').loc[list(PROBE_TEST_SESSION_IDS), 'error_rate']
    ax.bar(x + (offset - 1) * width, values, width=width, label=scheme_name)
ax.set_xticks(x, [session.replace('t12.2022.', '') for session in PROBE_TEST_SESSION_IDS])
ax.set_xlabel('validation session')
ax.set_ylabel('token error rate')
ax.set_title('Frozen GRU: native versus post-hoc grouped predictions')
ax.legend()
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'grouped_error_by_session.png', dpi=180, bbox_inches='tight')
plt.show()


## First probe: consonant articulator accessibility

For every reference consonant except `HH`, use the GRU logits to Viterbi-align the known phoneme sequence and average the hidden states across that aligned span. Fit three independent binary logistic probes for **lips**, **tongue front**, and **tongue body**. Multi-articulator phones such as `W` and `R` may be positive for more than one target. Vowels, `SIL`, and CTC blank do not enter this first probe.

The scaler and probe are fitted on the first 20 sessions only. The final four sessions are evaluated separately and pooled. A within-session label shuffle provides a diagnostic comparison while preserving each test session's class prevalence; because phoneme events within trials and sessions are correlated, its p-value is not a formal session-level inferential test. The probed hidden state is also the direct input to the GRU's trained linear phoneme head, so success shows that the coarse grouping can be read out from an already phoneme-trained representation—not that a new articulatory manifold has been discovered. The base GRU is frozen but not session-held-out: it learned session adapters from `competition_train` for all 24 sessions and was selected using the complete `competition_test` split. This is therefore a representation-accessibility result, not independent future-session decoding.


In [ ]:
from experiments.manifolds.articulator_probe import (
    DEFAULT_ARTICULATOR_TARGETS,
    build_aligned_consonant_events,
    fit_session_heldout_articulator_probes,
    load_articulatory_taxonomy,
    load_representation_arrays,
)

TAXONOMY_PATH = REPO_DIR / 'experiments/manifolds/design/articulatory_feature_taxonomy.csv'
PROBE_TARGETS = tuple(DEFAULT_ARTICULATOR_TARGETS)
PROBE_PERMUTATIONS = 1000
PROBE_SEED = 7
PROBE_MAX_ITERATIONS = 2000
PROBE_TOLERANCE = 1e-4

taxonomy = load_articulatory_taxonomy(TAXONOMY_PATH)
hidden, probe_logits, token_example_indices = load_representation_arrays(export_dir)
articulator_features, articulator_events = build_aligned_consonant_events(
    hidden=hidden,
    logits=probe_logits,
    token_example_indices=token_example_indices,
    examples=examples,
    taxonomy=taxonomy,
    blank_index=blank_index,
    targets=PROBE_TARGETS,
    excluded_symbols=('HH',),
)
assert set(articulator_events['session_id']) == set(ALL_VAL_SESSION_IDS)
assert not {'HH', 'SIL', 'BLANK'} & set(articulator_events['symbol'])
assert np.isfinite(articulator_features).all()
event_distribution = articulator_events.groupby('session_id').agg(
    events=('event_index', 'count'),
    lips=('lips', 'sum'),
    tongue_front=('tongue_front', 'sum'),
    tongue_body=('tongue_body', 'sum'),
).reset_index()
display(event_distribution)
print({'events': len(articulator_events), 'hidden_dim': articulator_features.shape[1]})


In [ ]:
probe_result = fit_session_heldout_articulator_probes(
    features=articulator_features,
    events=articulator_events,
    train_session_ids=PROBE_TRAIN_SESSION_IDS,
    test_session_ids=PROBE_TEST_SESSION_IDS,
    targets=PROBE_TARGETS,
    permutations=PROBE_PERMUTATIONS,
    seed=PROBE_SEED,
    max_iterations=PROBE_MAX_ITERATIONS,
    tolerance=PROBE_TOLERANCE,
)
probe_pooled = probe_result['pooled_metrics']
probe_by_session = probe_result['session_metrics']
probe_null = probe_result['null_metrics']
probe_predictions = probe_result['predictions']
assert probe_pooled['converged'].all()
display(probe_pooled)
display(probe_by_session.pivot(index='session_id', columns='target', values='balanced_accuracy').reset_index())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
target_x = np.arange(len(PROBE_TARGETS))
pooled_by_target = probe_pooled.set_index('target').loc[list(PROBE_TARGETS)]
axes[0].bar(target_x, pooled_by_target['balanced_accuracy'], width=0.6, label='probe')
axes[0].scatter(target_x, pooled_by_target['within_session_shuffle_mean_balanced_accuracy'], color='black', marker='x', label='within-session shuffle')
axes[0].axhline(0.5, color='gray', linestyle='--', linewidth=1, label='balanced chance')
axes[0].set_xticks(target_x, PROBE_TARGETS, rotation=20)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('balanced accuracy')
axes[0].set_title('Pooled final-four-session evaluation')
axes[0].legend(fontsize=8)

width = 0.24
session_x = np.arange(len(PROBE_TEST_SESSION_IDS))
for offset, target in enumerate(PROBE_TARGETS):
    values = probe_by_session[probe_by_session.target == target].set_index('session_id').loc[list(PROBE_TEST_SESSION_IDS), 'balanced_accuracy']
    axes[1].bar(session_x + (offset - 1) * width, values, width=width, label=target)
axes[1].axhline(0.5, color='gray', linestyle='--', linewidth=1)
axes[1].set_xticks(session_x, [session.replace('t12.2022.', '') for session in PROBE_TEST_SESSION_IDS])
axes[1].set_ylim(0, 1)
axes[1].set_xlabel('held-out probe session')
axes[1].set_ylabel('balanced accuracy')
axes[1].set_title('Per-session stability')
axes[1].legend(fontsize=8)
fig.savefig(OUTPUT_DIR / 'articulator_probe_balanced_accuracy.png', dpi=180, bbox_inches='tight')
plt.show()


## How to decide whether to retrain

Focus first on `broad_3way` error across all four sessions. If it is low and stable enough for the intended descriptive analysis, retain the frozen native GRU and sum its phoneme posteriors into groups. If broad errors remain substantial or one session fails badly, a coarse-label retraining experiment is justified. The manner score tells us whether the more interesting articulatory subdivision is already usable.

For the articulator probe, look for balanced accuracy above the within-session shuffle distribution for all three targets and broadly consistent performance in each of the four test sessions. Treat the shuffle distribution as a diagnostic, not formal session-level inference. A strong result means articulator membership is linearly accessible in the trained decoder state. It does not establish that the same separation exists in raw SBP, and it is expected to be easier than probing raw activity because the GRU was trained for phoneme decoding.

Neither test establishes timing accuracy. T12 has no independent frame labels with which to validate the GRU's 80 ms timing. A positive result supports using the model as a source of soft, explicitly model-assisted timing—not treating its boundaries as ground truth.


In [ ]:
from datetime import datetime, timezone
import hashlib

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
group_definition.to_csv(OUTPUT_DIR / 'phoneme_groups.csv', index=False)
trial_metrics.to_csv(OUTPUT_DIR / 'trial_metrics.csv', index=False)
per_session_metrics.to_csv(OUTPUT_DIR / 'per_session_metrics.csv', index=False)
pooled_metrics.to_csv(OUTPUT_DIR / 'pooled_metrics.csv', index=False)
posterior_summary.to_csv(OUTPUT_DIR / 'posterior_summary.csv', index=False)
pd.read_csv(TAXONOMY_PATH, keep_default_na=False).to_csv(OUTPUT_DIR / 'articulatory_feature_taxonomy.csv', index=False)
articulator_events.to_csv(OUTPUT_DIR / 'articulator_events.csv', index=False)
event_distribution.to_csv(OUTPUT_DIR / 'articulator_event_distribution.csv', index=False)
probe_pooled.to_csv(OUTPUT_DIR / 'articulator_probe_pooled.csv', index=False)
probe_by_session.to_csv(OUTPUT_DIR / 'articulator_probe_by_session.csv', index=False)
probe_null.to_csv(OUTPUT_DIR / 'articulator_probe_null.csv', index=False)
probe_predictions.to_csv(OUTPUT_DIR / 'articulator_probe_predictions.csv', index=False)
probe_parameter_arrays = {}
for target, pipeline in probe_result['models'].items():
    scaler = pipeline.named_steps['scale']
    classifier = pipeline.named_steps['probe']
    probe_parameter_arrays[f'{target}_scaler_mean'] = scaler.mean_
    probe_parameter_arrays[f'{target}_scaler_scale'] = scaler.scale_
    probe_parameter_arrays[f'{target}_coef_standardized'] = classifier.coef_
    probe_parameter_arrays[f'{target}_intercept'] = classifier.intercept_
    probe_parameter_arrays[f'{target}_classes'] = classifier.classes_
    probe_parameter_arrays[f'{target}_n_iter'] = classifier.n_iter_
np.savez_compressed(OUTPUT_DIR / 'articulator_probe_parameters.npz', **probe_parameter_arrays)
taxonomy_sha256 = hashlib.sha256(TAXONOMY_PATH.read_bytes()).hexdigest()
provenance = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'checkpoint_contract': checkpoint_contract,
    'export_metadata_path': str(export_dir / 'metadata.json'),
    'evaluation_note': 'competition_test labels participated in checkpoint selection; exploratory validation diagnostic, not an untouched test',
    'grouping_note': 'native CTC decode first; map tokens second; preserve adjacent repeated groups; keep CTC blank separate from SIL',
    'probe_contract': {
        'targets': list(PROBE_TARGETS),
        'feature': 'mean frozen GRU hidden state over reference-constrained CTC span',
        'excluded_symbols': ['HH', 'SIL', 'BLANK', 'all vowels'],
        'train_sessions': list(PROBE_TRAIN_SESSION_IDS),
        'test_sessions': list(PROBE_TEST_SESSION_IDS),
        'classifier': 'StandardScaler plus class-weighted LogisticRegression(C=1.0, liblinear); solver stops when tolerance is met',
        'permutations': PROBE_PERMUTATIONS,
        'seed': PROBE_SEED,
        'max_iterations': PROBE_MAX_ITERATIONS,
        'tolerance': PROBE_TOLERANCE,
        'taxonomy_path': str(TAXONOMY_PATH),
        'taxonomy_sha256': taxonomy_sha256,
    },
    'probe_exposure_note': 'probe head is session-held-out; frozen GRU adapters and checkpoint selection are not session-held-out',
    'readout_note': 'the probed hidden state directly feeds the trained linear phoneme head, so success is coarse-label accessibility within a phoneme-trained representation, not discovery of an independent articulatory manifold',
    'shuffle_note': 'within-session event-label permutations are a diagnostic comparison, not formal session-level inference because events are correlated within trials and sessions',
    'provenance_note': 'local GRU is an LLM-assisted adaptation of likely Willett/Stanford TensorFlow source; exact upstream origin remains unresolved',
    'ai_assistance': 'Codex implemented the taxonomy-backed alignment, probe, validation, and notebook workflow; human review is required before scientific use.',
}
(OUTPUT_DIR / 'provenance.json').write_text(json.dumps(provenance, indent=2))

for required_name in (
    'phoneme_groups.csv', 'trial_metrics.csv', 'per_session_metrics.csv',
    'pooled_metrics.csv', 'posterior_summary.csv', 'provenance.json',
    'grouped_error_by_session.png',
    'articulatory_feature_taxonomy.csv', 'articulator_events.csv',
    'articulator_event_distribution.csv', 'articulator_probe_pooled.csv',
    'articulator_probe_by_session.csv', 'articulator_probe_null.csv',
    'articulator_probe_predictions.csv', 'articulator_probe_parameters.npz',
    'articulator_probe_balanced_accuracy.png',
):
    path = OUTPUT_DIR / required_name
    assert path.exists() and path.stat().st_size > 0, path
reopened = pd.read_csv(OUTPUT_DIR / 'pooled_metrics.csv')
assert set(reopened['scheme']) == set(schemes)
assert np.isfinite(reopened['error_rate']).all()
probe_reopened = pd.read_csv(OUTPUT_DIR / 'articulator_probe_pooled.csv')
session_reopened = pd.read_csv(OUTPUT_DIR / 'articulator_probe_by_session.csv')
assert set(probe_reopened['target']) == set(PROBE_TARGETS)
assert set(session_reopened['session_id']) == set(PROBE_TEST_SESSION_IDS)
assert np.isfinite(probe_reopened['balanced_accuracy']).all()
assert probe_reopened['converged'].all()
with np.load(OUTPUT_DIR / 'articulator_probe_parameters.npz') as saved_parameters:
    assert set(saved_parameters.files) == set(probe_parameter_arrays)
RESULT_COMPLETION_MARKER.write_text(provenance['created_utc'])
assert RESULT_COMPLETION_MARKER.read_text() == provenance['created_utc']
print('Artifact reopen checks passed.')
print('Saved:', OUTPUT_DIR)
display(reopened)
display(probe_reopened)


In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()
